# Week 2. Counting Is Already a Model

**What you'll do today.** You'll build a word-counter by hand-logic, watch the same text get
counted three different ways, scale your count up with tf-idf, run the same counter across three
very different corpora, upgrade the signature-vocabulary trick into proper **keyness** (the
method behind Week 1's featured study), and finish with the one statistics move this course
needs: the **shuffle test** for whether a counted difference is real. (You already counted an
*image*, pixel by pixel, in Week 1; this week goes deep on words.)

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib pillow

In [ ]:
# Imports. If this cell fails, it almost always means the install cell above didn't run
# this session, re-run it, then re-run this. (See the cheat sheet, entry 5.)
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
print("imports ok")

In [ ]:
# Are we running tiny + offline (the compatibility harness sets this), or in a real
# Colab class session? Everything below adapts so the notebook runs either way.
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Three little corpora

A *corpus* is just "a pile of text you're studying." We'll start with the one you live in, a
slice of pop lyrics, then a subreddit slice, then a public-domain novel.

In [ ]:
# Small, self-contained stand-ins so the notebook runs anywhere. In class, the AI
# replaces these with live pulls (a lyrics slice, a subreddit export, a Gutenberg novel).
lyrics = [
    "we were both young when I first saw you I close my eyes",
    "cause baby now we got bad blood you know it used to be mad love",
    "it's a cruel summer with you and I burning",
    "shake it off shake it off I shake it off",
    "you call me up again just to break me like a promise",
]
reddit = [
    "AITA for not going to my sister's wedding after she uninvited my partner",
    "AITA for telling my roommate the dishes are not going to wash themselves",
    "AITA for leaving work early when my boss kept moving the deadline",
    "update I talked to my sister and we are okay now thanks everyone",
]
novel = (
    "It is a truth universally acknowledged that a single man in possession of a good "
    "fortune must be in want of a wife. However little known the feelings or views of "
    "such a man may be on his first entering a neighbourhood this truth is so well fixed."
)
corpora = {"lyrics": lyrics, "reddit": reddit, "novel": [novel]}
for name, docs in corpora.items():
    print(f"{name:8s}: {len(docs)} documents")

## Bag-of-words by hand: counting *requires defining*

Before any library, count by hand-logic.

In [ ]:
def tokenize(text, fold_case=True, strip_punct=True):
    if fold_case:
        text = text.lower()
    if strip_punct:
        text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [t for t in text.split() if t]

all_lyrics = " ".join(lyrics)
counts = Counter(tokenize(all_lyrics))
print("Top words in the lyrics slice:")
for word, n in counts.most_common(8):
    print(f"  {word:8s} {n}")

In [ ]:
# A "stop word" list is itself a choice. Watch the top-words list change when we drop
# common function words, the same kind of decision as merging run/running.
STOP = {"i","we","you","it","the","a","and","to","of","in","so","be","on","my","me","now"}
counts_nostop = Counter(t for t in tokenize(all_lyrics) if t not in STOP)
print("Top words with stop-words removed:")
for word, n in counts_nostop.most_common(8):
    print(f"  {word:8s} {n}")

### What counts as a word? Tokens, not words

Models never see words.

## tf-idf: "common here, rare overall"

Raw counts are dominated by words that are common *everywhere* (*the*, *you*).

In [ ]:
docs = lyrics + reddit + [novel]
labels = ["lyric"]*len(lyrics) + ["reddit"]*len(reddit) + ["novel"]
vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform(docs)
terms = np.array(vec.get_feature_names_out())

# The most distinctive term in each lyric line:
for i in range(len(lyrics)):
    row = X[i].toarray().ravel()
    top = terms[row.argmax()]
    print(f"lyric {i}: most distinctive word -> '{top}'")

## Cross-corpus counting: the corpus choice changes what "common" means

Same counter, three corpora.

In [ ]:
for name, dd in corpora.items():
    text = " ".join(dd)
    top = Counter(t for t in tokenize(text) if t not in STOP).most_common(5)
    print(f"{name:8s}:", ", ".join(f"{w}({n})" for w, n in top))

### Delight beat, a signature vocabulary

Counting alone can show you a *voice*: the words one source uses far more than a baseline.

In [ ]:
# Words the lyrics use much more than the "baseline" (reddit + novel), by simple rate ratio.
def rates(text):
    toks = tokenize(text)
    n = len(toks) or 1
    return {w: c/n for w, c in Counter(toks).items()}

sig = rates(" ".join(lyrics))
base = rates(" ".join(reddit) + " " + novel)
distinctive = sorted(((w, sig[w]/(base.get(w,0)+1e-6)) for w in sig if w not in STOP),
                     key=lambda kv: kv[1], reverse=True)[:6]
print("Lyrics' signature words vs. baseline:")
for w, r in distinctive:
    print(f"  {w}")

## Keyness: distinctively hers, done properly

The ratio above was a rough cut. The standard tool is a **log-odds ratio**: for every word,
how much more likely is it in corpus A than corpus B (with a little smoothing so rare words
don't explode)? Positive means "distinctively A," negative "distinctively B."

**The reveal:** this is *exactly* the method behind Week 1's featured study. *She Giggles, He
Gallops* is a log-odds ratio of stage-direction verbs after "she" vs. "he," run on 2,000
screenplays. You now own the arithmetic behind the piece you admired.

In [ ]:
import numpy as np
from collections import Counter

def counts(texts):
    c = Counter()
    for t in texts:
        c.update(tokenize(t))
    return c

A, B = counts(lyrics), counts(reddit + [novel])          # lyrics vs. everything-else baseline
vocab = [w for w in (A | B) if (A[w] + B[w]) >= 3]        # ignore the rarest words
NA, NB = sum(A.values()), sum(B.values())

def log_odds(w):
    pa = (A[w] + 1) / (NA + len(vocab))                   # +1 smoothing
    pb = (B[w] + 1) / (NB + len(vocab))
    return np.log(pa / (1 - pa)) - np.log(pb / (1 - pb))

scored = sorted(vocab, key=log_odds)
print("distinctively LYRICS: ", [w for w in scored[-8:][::-1]])
print("distinctively BASELINE:", [w for w in scored[:8]])

## Is the difference real? The shuffle test

Your top keyness word is more common in the lyrics than in the baseline. But small corpora
produce flukes. The honest check needs no formulas: **shuffle the labels and count again**.
If documents were randomly dealt into "lyrics" and "baseline" piles a thousand times, how
often would chance alone produce a gap as big as the real one? Rarely: you have a finding.
Often: you have a coincidence. (And a gap can be real *and* tiny; the shuffle test says
"probably not chance," never "big enough to matter." That's your call to argue.)

In [ ]:
rng = np.random.default_rng(0)
word = scored[-1]                                          # our most "distinctively lyrics" word
docs_all = lyrics + reddit + [novel]
labels = [1]*len(lyrics) + [0]*(len(reddit) + 1)           # 1 = lyrics, 0 = baseline

def rate_gap(labels):
    a = [d for d, l in zip(docs_all, labels) if l]
    b = [d for d, l in zip(docs_all, labels) if not l]
    r = lambda texts: sum(tokenize(t).count(word) for t in texts) / max(1, sum(len(tokenize(t)) for t in texts))
    return r(a) - r(b)

observed = rate_gap(labels)
shuffled = []
for _ in range(1000):
    rng.shuffle(labels)
    shuffled.append(rate_gap(labels))

beat = sum(abs(g) >= abs(observed) for g in shuffled)
plt.figure(figsize=(6, 3.5))
plt.hist(shuffled, bins=30)
plt.axvline(observed, color="red", linewidth=2, label=f"the real gap for {word!r}")
plt.title(f"1,000 shuffles: chance matched the real gap {beat} times")
plt.xlabel("gap produced by a random shuffle"); plt.ylabel("shuffles")
plt.legend(); plt.tight_layout(); plt.show()
print(f"chance beat the observed gap in {beat}/1000 shuffles",
      "- probably not chance." if beat < 50 else "- could easily be chance; don't lean on it.")

## You made a counter, and you can defend it

You defined what a word is, counted three corpora three ways, found the words that make a
voice distinctive with the same method as a famous published study, and shuffle-tested the
gap so you know it isn't chance.

**Sketch (homework):** count something in a text you love; make one chart; write one sentence
naming a choice you made. If your count compares two things, shuffle-test the gap.